<a href="https://colab.research.google.com/github/IshiPareek/mech_interpet/blob/main/04_gpt2_induction_head.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — imports
!pip install transformer_lens
import transformer_lens
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
import torch
import numpy as np
from jaxtyping import Float, Int
import einops

print(f"TransformerLens installd")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")

In [ ]:
token1 = model.to_tokens("The meaning of life is")
str_tokens = model.to_str_tokens(token1)
print(model.to_str_tokens(token1))
logits = model(token1)
print(logits.shape)

token2 = model.to_tokens("The capital of France is ")
str_tokens = model.to_str_tokens(token2)
print(model.to_str_tokens(token2))
logits = model(token2)
print(logits.shape)

In [ ]:
# ── VISUALIZATION BOILERPLATE ── define once, call anywhere ──────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def viz(cache, plot_type, layer=0, head=0, dims=64, model=model, str_tokens=str_tokens, original_loss=None, ablated_loss=None):
    """
    Universal cache visualizer.

    plot_type options:
        "embed"      → token embeddings
        "pos_embed"  → positional embeddings
        "pattern"    → attention pattern for one head
        "all_heads"  → all 12 heads in a layer
        "resid"      → residual stream at a layer
        "logit_lens" → top predicted token across all layers
        "qkv"        → query, key, value vectors
    """

    def heatmap(ax, matrix, title, row_labels=None, cmap="RdBu_r"):
        vmax = np.abs(matrix).max()
        ax.imshow(matrix, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=10, pad=6)
        if row_labels:
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=8)
        else:
            ax.set_yticks([])
        ax.set_xticks([])

    # ── embed ────────────────────────────────────────────────────────────────
    if plot_type == "embed":
        data = cache["hook_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_embed — first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── pos_embed ────────────────────────────────────────────────────────────
    elif plot_type == "pos_embed":
        data = cache["hook_pos_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_pos_embed — first {dims} dims", row_labels=str_tokens, cmap="PiYG")
        plt.tight_layout(); plt.show()

    # ── pattern ──────────────────────────────────────────────────────────────
    elif plot_type == "pattern":
        data = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().numpy()
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(str_tokens))); ax.set_xticklabels(str_tokens, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(str_tokens))); ax.set_yticklabels(str_tokens, fontsize=8)
        ax.set_title(f"attention pattern — layer {layer}, head {head}", fontsize=10)
        plt.tight_layout(); plt.show()

    # ── all_heads ────────────────────────────────────────────────────────────
    elif plot_type == "all_heads":
        fig, axes = plt.subplots(3, 4, figsize=(14, 9))
        for h, ax in enumerate(axes.flat):
            data = cache[f"blocks.{layer}.attn.hook_pattern"][0, h].detach().numpy()
            ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"H{h}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(f"all attention heads — layer {layer}", fontsize=12)
        plt.tight_layout(); plt.show()

    # ── resid ────────────────────────────────────────────────────────────────
    elif plot_type == "resid":
        data = cache[f"blocks.{layer}.hook_resid_post"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_resid_post — layer {layer}, first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── logit_lens ───────────────────────────────────────────────────────────
    elif plot_type == "logit_lens":
        n_layers = model.cfg.n_layers
        cell_text = []
        for l in range(n_layers):
            resid = cache[f"blocks.{l}.hook_resid_post"]
            top = model.unembed(model.ln_final(resid))[0, :, :].argmax(dim=-1)
            cell_text.append([model.to_string(t) for t in top])
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.axis("off")
        tbl = ax.table(cellText=cell_text, rowLabels=[f"L{l}" for l in range(n_layers)],
                       colLabels=str_tokens, loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
        ax.set_title("logit lens — top predicted token per layer × position", fontsize=11, pad=20)
        plt.tight_layout(); plt.show()

    # ── qkv ──────────────────────────────────────────────────────────────────
    elif plot_type == "qkv":
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        for ax, key, label in zip(axes, ["hook_q","hook_k","hook_v"], ["Query","Key","Value"]):
            data = cache[f"blocks.{layer}.attn.{key}"][0, :, head, :].detach().numpy()
            heatmap(ax, data, f"{label} — layer {layer}, head {head}", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

## ABLATION
    elif plot_type == "ablation":
      if original_loss is None or ablated_loss is None :
        print ("pass original_loss and ablation_loss to use this plot type")
        return

      orig = original_loss.item()
      abla = ablated_loss.item()
      diff = abla - orig

      fig, ax = plt.subplots(figsize=(5, 4))
      bars = ax.bar(
      ["Original Loss", "Ablated Loss"],
      [orig, abla],
      color=["steelblue", "tomato"])

      ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
      ax.set_title(
            f"Head Ablation — Layer {layer}, Head {head}\nΔ Loss = +{diff:.3f}",
      fontsize=11)
      ax.set_ylabel("Cross Entropy Loss")
      ax.set_ylim(0, max(orig, abla) * 1.2)
      plt.tight_layout(); plt.show()

    else:
        print(f"unknown plot_type '{plot_type}'. options: embed, pos_embed, pattern, all_heads, resid, logit_lens, qkv")

Induction Heads

Imp : Detect and continue repeated subsequences

Consist of 2 heads :
1/ Prev token head : Attends to the prev token

2/ Induction head : token after an earlier copy of the token

Basically, looks at what absolute truth it saw  for before and predicts the same thing.

In [ ]:
batch_size = 10
seq_len = 50
size = (batch_size, seq_len)
input_tensor = torch.randint(1000, 10000, size)

random_tokens = input_tensor.to(model.cfg.device)
repeated_tokens = einops.repeat(random_tokens, "batch seq_len -> batch (2 seq_len)")
repeated_logits = model(repeated_tokens)
correct_log_probs = model.loss_fn(repeated_logits, repeated_tokens, per_token=True)
loss_by_position = einops.reduce(correct_log_probs, "batch position -> position", "mean")

#Plot
plt.figure(figsize=(12, 4))
plt.plot(loss_by_position.detach().cpu().numpy())
plt.axvline(x=50, color='red', linestyle='--', label='repetition starts')
plt.xlabel("Position")
plt.ylabel("Loss")
plt.title("Loss by position on random repeated tokens")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
batch_size = 10
seq_len = 50
size = (batch_size, seq_len)
input_tensor = torch.randint(1000, 10000, size)

random_tokens = input_tensor.to(model.cfg.device)
repeated_tokens = einops.repeat(random_tokens, "batch seq_len -> batch (2 seq_len)")
repeated_logits = model(repeated_tokens)

# per token loss — don't reduce across batch this time
correct_log_probs = model.loss_fn(repeated_logits, repeated_tokens, per_token=True)
# shape: (batch, position) = (10, 99)

# plot each batch row as a heatmap
plt.figure(figsize=(16, 4))
plt.imshow(
    correct_log_probs.detach().cpu().numpy(),
    cmap="RdBu_r",
    aspect="auto",
    vmin=0,
    vmax=15
)
plt.colorbar(label="Loss")
plt.axvline(x=49, color='yellow', linestyle='--', linewidth=2, label='repetition starts')
plt.xlabel("Position")
plt.ylabel("Batch")
plt.title("Per-token Loss — all 10 sequences\n(blue = low loss, red = high loss)")
plt.legend()
plt.tight_layout()
plt.show()

What if we tired to see not what the induction head does, but where does it start kicking in throughout the layers? See below for an implementation

In [ ]:
batch_size = 10
seq_len = 50
input_tensor = torch.randint(1000, 10000, (batch_size, seq_len))
random_tokens = input_tensor.to(model.cfg.device)
repeated_tokens = einops.repeat(random_tokens, "batch seq_len -> batch (2 seq_len)")

# run with cache to get all intermediate activations
_, cache = model.run_with_cache(repeated_tokens)

# for each layer, project residual stream → logits and measure loss
layer_losses_first = []   # first half (random)
layer_losses_second = []  # second half (repeated)

for layer in range(model.cfg.n_layers):
    # get residual stream at this layer
    resid = cache[f"blocks.{layer}.hook_resid_post"]

    # project to logits using final layernorm + unembed
    logits = model.unembed(model.ln_final(resid))

    # compute loss per token
    log_probs = model.loss_fn(logits, repeated_tokens, per_token=True)
    # shape: (batch, position)

    # average loss for first half vs second half
    first_half  = log_probs[:, :seq_len-1].mean().item()
    second_half = log_probs[:, seq_len:].mean().item()

    layer_losses_first.append(first_half)
    layer_losses_second.append(second_half)

# plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(layer_losses_first,  label="first half (random)", color="tomato",    marker="o")
ax.plot(layer_losses_second, label="second half (repeated)", color="steelblue", marker="o")
ax.set_xlabel("Layer")
ax.set_ylabel("Average Loss")
ax.set_title("Loss by layer — where does induction kick in?\nlogit lens on repeated tokens")
ax.set_xticks(range(model.cfg.n_layers))
ax.set_xticklabels([f"L{l}" for l in range(model.cfg.n_layers)])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# We make a tensor to store the induction score for each head. We put it on the model's device to avoid needing to move things between the GPU and CPU, which can be slow.
induction_score_store = torch.zeros(
    (model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device
)


def induction_score_hook(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    # We take the diagonal of attention paid from each destination position to source positions seq_len-1 tokens back
    # (This only has entries for tokens with index>=seq_len)
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    # Get an average score per head
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    # Store the result.
    induction_score_store[hook.layer(), :] = induction_score


# We make a boolean filter on activation names, that's true only on attention pattern names.
pattern_hook_names_filter = lambda name: name.endswith("pattern")

model.run_with_hooks(
    repeated_tokens,
    return_type=None,  # For efficiency, we don't need to calculate the logits
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)],
)

#PLOT
plt.figure(figsize=(10, 6))
plt.imshow(induction_score_store.detach().cpu().numpy(), cmap="Blues", aspect="auto")
plt.colorbar(label="Induction Score")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.title("Induction Score by Head")
plt.xticks(range(model.cfg.n_heads), [f"H{h}" for h in range(model.cfg.n_heads)])
plt.yticks(range(model.cfg.n_layers), [f"L{l}" for l in range(model.cfg.n_layers)])
plt.tight_layout()
plt.show()